# L3: Supervised Fine-Tuning (SFT)

In [11]:
# Warning control
import warnings
warnings.filterwarnings('ignore')
import os
import gc

## Import libraries

In [12]:
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import TrainingArguments, AutoTokenizer, AutoModelForCausalLM
#from modelscope import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig
from torch.cuda.amp import autocast  # 对于支持 CUDA 的设备，使用 amp
# Memory optimization imports
import torch.nn.utils.prune as prune
from typing import Optional

## Setting up helper functions

In [13]:
def clear_memory():
    """Clear GPU/MPS memory and run garbage collection"""
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    gc.collect()

def get_memory_usage():
    """Get current memory usage"""
    if torch.backends.mps.is_available():
        allocated = torch.mps.current_allocated_memory() / 1024**3  # GB
        reserved = torch.mps.driver_allocated_memory() / 1024**3   # GB
        print(f"MPS Memory - Allocated: {allocated:.2f}GB, Reserved: {reserved:.2f}GB")
        return allocated, reserved
    return 0, 0

def setup_memory_efficient_training():
    """Setup memory efficient training environment"""
    # Set environment variables for memory optimization
    os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'  # Disable memory caching
    os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'  # Enable fallback to CPU
    
    # Clear memory at start
    clear_memory()

# %% [markdown]
# ## Setting up helper functions (optimized)

def generate_responses(model, tokenizer, user_message, system_message=None, 
                       max_new_tokens=50):  # Reduced from 100 to save memory
    """Generate response with memory optimization"""
    # Format chat using tokenizer's chat template
    messages = []
    if system_message:
        messages.append({"role": "system", "content": system_message})
    
    messages.append({"role": "user", "content": user_message})
        
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    
    # Move to device only when needed
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        # Use smaller memory footprint generation
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=False,  # Disable KV cache to save memory
        )
    
    input_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_len:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    # Clear intermediate tensors
    del inputs, outputs, generated_ids
    clear_memory()
    
    return response

In [14]:
def test_model_with_questions(model, tokenizer, questions, 
                              system_message=None, title="Model Output"):
    print(f"\n=== {title} ===")
    for i, question in enumerate(questions, 1):
        try:
            response = generate_responses(model, tokenizer, question, 
                                          system_message)
            print(f"\nModel Input {i}:\n{question}\nModel Output {i}:\n{response}\n")
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"Memory error for question {i}, skipping...")
                clear_memory()
            else:
                raise e


In [15]:

def load_model_and_tokenizer(model_name, use_gpu=False):
    """Load model with memory optimization options"""
    
    print(f"Loading model: {model_name}")
    
    # Load tokenizer first (smaller memory footprint)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Load model with optimizations
    model_kwargs = {
        "torch_dtype": torch.float32,  # Use half precision
        "low_cpu_mem_usage": True,     # Reduce CPU memory usage during loading
    }
    
    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    
    # Move to appropriate device
    if use_gpu and torch.backends.mps.is_available():
        model.to("mps")
        print("Model moved to MPS")
    elif use_gpu and torch.cuda.is_available():
        model.to("cuda")
        print("Model moved to CUDA")
    else:
        print("Using CPU")
    
    # Setup chat template if not exists
    if not tokenizer.chat_template:
        tokenizer.chat_template = """{% for message in messages %}
                {% if message['role'] == 'system' %}System: {{ message['content'] }}\n
                {% elif message['role'] == 'user' %}User: {{ message['content'] }}\n
                {% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }} <|endoftext|>
                {% endif %}
                {% endfor %}"""
    
    # Tokenizer config
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token
        
    return model, tokenizer

In [16]:
# %%
def display_dataset(dataset, num_samples=2):  # Reduced from 3 to 2
    """Display dataset samples with memory optimization"""
    rows = []
    for i in range(min(num_samples, len(dataset))):
        example = dataset[i]
        user_msg = next(m['content'] for m in example['messages']
                        if m['role'] == 'user')
        assistant_msg = next(m['content'] for m in example['messages']
                             if m['role'] == 'assistant')
        
        # Truncate long messages for display
        user_msg = user_msg[:200] + "..." if len(user_msg) > 200 else user_msg
        assistant_msg = assistant_msg[:200] + "..." if len(assistant_msg) > 200 else assistant_msg
        
        rows.append({
            'User Prompt': user_msg,
            'Assistant Response': assistant_msg
        })
    
    df = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', 100)  # Limit column width
    display(df)

## Load base model & test on simple questions

In [17]:
setup_memory_efficient_training()

# Use smaller model and reduced settings for M4 Pro
USE_GPU = True  # Enable MPS usage
LOAD_IN_8BIT = False  # Disable 8bit for compatibility

questions = [
    "Give me a 1-sentence introduction of LLM.",
    "Calculate 1+1-1",
    "What's the difference between thread and process?"
]
# Check device availability
print("Device availability check:")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"CUDA available: {torch.cuda.is_available()}")

device = torch.device("mps" if torch.backends.mps.is_available() else 
                     ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Using device: {device}")

get_memory_usage()

Device availability check:
MPS available: True
CUDA available: False
Using device: mps
MPS Memory - Allocated: 31.44GB, Reserved: 44.03GB


(31.4395809173584, 44.033416748046875)

In [18]:
base_model_name = "Qwen/Qwen3-4B-Base"  # Much smaller model for testing
model, tokenizer = load_model_and_tokenizer(base_model_name, USE_GPU)

get_memory_usage()

test_model_with_questions(model, tokenizer, questions, 
                          title="Base Model (Before SFT) Output")

# Clean up
del model, tokenizer
clear_memory()

Loading model: Qwen/Qwen3-4B-Base


Loading checkpoint shards: 100%|██████████| 3/3 [00:17<00:00,  5.98s/it]


Model moved to MPS
MPS Memory - Allocated: 46.42GB, Reserved: 58.12GB

=== Base Model (Before SFT) Output ===

Model Input 1:
Give me a 1-sentence introduction of LLM.
Model Output 1:
LLMs, or Large Language Models, are advanced AI systems trained on vast amounts of text data to generate human-like text, understand context, and perform a wide range of natural language tasks, from translation to creative writing.


Model Input 2:
Calculate 1+1-1
Model Output 2:
:convert 1+1-1:convert
:convert :convert
:convert:convert
:convert:convert
:convert:convert
:convert:convert
:convert:convert
:convert:convert
:convert:convert
:convert:convert
:convert:convert
:convert:convert
:convert:convert
:convert:convert
:convert


Model Input 3:
What's the difference between thread and process?
Model Output 3:
Threads and processes are both mechanisms for concurrent execution in a computer program, but they differ in several key aspects:

### 1. **Definition:**
   - **Process:** A process is an instance o

## Doing SFT on a small model

In [19]:
# Use an even smaller model for SFT training
model_name = "Qwen/Qwen3-4B-Base"
model, tokenizer = load_model_and_tokenizer(model_name, USE_GPU)

get_memory_usage()

print(*(load_dataset("banghua/DL-SFT-Dataset").keys()))
# %%
# Load and prepare training dataset with heavy optimization
train_dataset = load_dataset("banghua/DL-SFT-Dataset")["train"]
print(f"full train set length:{len(train_dataset)}")

# Drastically reduce dataset size for M4 Pro
#if USE_GPU:
# train_dataset = train_dataset.select(range(500))  # Only 100 samples
#else:
#    train_dataset = train_dataset.select(range(100))   # Only 50 samples for CPU

print(f"Training dataset size: {len(train_dataset)}")
display_dataset(train_dataset)

Loading model: Qwen/Qwen3-4B-Base


Loading checkpoint shards: 100%|██████████| 3/3 [00:06<00:00,  2.21s/it]


Model moved to MPS
MPS Memory - Allocated: 46.42GB, Reserved: 58.12GB
train
full train set length:2961
Training dataset size: 2961


,User Prompt,Assistant Response
0,"- The left child should have a value less than the parent node's value, and the right child shou...","This statement is correct. In a binary search tree, nodes in the left subtree of a particular no..."
1,"To pass three levels must be the plan.\nThen tackle Two, when that is done.\nOf 100 that start, ...","Based on the given information, out of 100 individuals who start, only 20 will make it to the en..."


In [ ]:
# Heavily optimized SFTTrainer config for M4 Pro
sft_config = SFTConfig(
    learning_rate=8e-5,                    # Reduced learning rate
    num_train_epochs=1,                    # Keep at 1 epoch
    per_device_train_batch_size=1,         # Keep batch size at 1
    gradient_accumulation_steps=8,          # Reduced from 16 to 4
    gradient_checkpointing=False,           # Enable gradient checkpointing
    logging_steps=10,                      # Less frequent logging
    bf16=False,                            # Keep bf16 disabled
    fp16=False,                            # Disable fp16 for MPS compatibility
    tf32=False,
    dataloader_pin_memory=False,           # Disable pin memory
    remove_unused_columns=True,            # Remove unused columns
    dataset_text_field="text",             # Specify text field
    packing=False,                         # Disable packing
    save_strategy="no",                    # Disable saving to save space
    eval_strategy="no",                    # Disable evaluation
    warmup_steps=5,                        # Reduced warmup steps
    optim="adamw_torch",                   # Use PyTorch AdamW
    lr_scheduler_type="linear",            # Linear scheduler
    seed=42,                               # Set seed for reproducibility
)
# Create trainer with memory optimization
try:
    sft_trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_dataset, 
        processing_class=tokenizer,
    )
    
    print("Starting training...")
    get_memory_usage()
    
    # Train with memory monitoring
    sft_trainer.train()
    
    print("Training completed!")
    get_memory_usage()
    
except RuntimeError as e:
    if "out of memory" in str(e):
        print(f"Out of memory error: {e}")
        print("Try reducing batch size, dataset size, or model size further")
        clear_memory()
    else:
        raise e
print("Testing trained model...")
get_memory_usage()
sft_trainer.save_model(f"./model/{model_name}-SFT")

Starting training...
MPS Memory - Allocated: 46.42GB, Reserved: 58.12GB


## Testing training results on small model and small dataset

**Note:** The following results are for the small model and dataset we used for SFT training, due to limited computational resources. To view the results of full-scale training on a larger model, see the **"SFT Results on Qwen3-0.6B Model"** section above.

In [ ]:
print("Testing trained model...")
get_memory_usage()

try:
    if USE_GPU:
        sft_trainer.model.to("mps")  # Move to CPU for inference to save MPS memory
        clear_memory()
    
    test_model_with_questions(sft_trainer.model, tokenizer, questions, 
                              title="Model (After SFT) Output")
except Exception as e:
    print(f"Testing error: {e}")
    print("Model testing skipped due to memory constraints")

# Final cleanup
clear_memory()
get_memory_usage()
